In [93]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from coffea.util import load

# Set the CMS style globally
hep.style.use(hep.style.CMS)

# Define the directory where your .coffea files are stored
COFFEA_DIR = "DataVsMC/" # <-- UPDATE THIS PATH
LUMI = 110 # in fb^-1 

# The list of variables you want to plot
validation_vars = ['nPV', 'nPVGood', 'MET_pt', 'MET_phi', 'lep_pt', 'ele_pt', 'muon_pt',
                   'lep_eta', 'ele_eta', 'muon_eta', 'n_ak4', 'n_bjet', 'n_ak8',
                   'jet1_pt', 'jet2_pt', 'bjet1_pt', 'bjet2_pt', 'jet1_eta', 'jet2_eta',
                   'bjet1_eta', 'bjet2_eta', 'jet1_btag', 'jet2_btag', 'bjet1_btag', 'bjet2_btag',
                   'fatjet1_pt', 'fatjet1_eta', 'fatjet1_mass', 'n_b_outZH', 'n_ak4jets']

In [94]:
bkg_processes = ['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX'] 
#bkg_processes = ['tt_B', 'TTBar'] 
sig_processes = ['ttZ', 'ttH']
data_process = 'data_obs'
all_processes = bkg_processes + sig_processes + [data_process]

# Updated with the new color scheme
bkg_colors = {
    'VJets': '#3f90da', 
    'QCD': '#ffa90e', 
    'tt_B': '#bd1f01', 
    'TTBar': '#94a4a2',
    'SingleTop': '#e76300',
    'TTX': '#b9ac70'
}
sig_colors = {
    'ttZ': '#832db6', 
    'ttH': '#a96b59'
}

process_labels = {
    'VJets': 'V+Jets',
    'QCD': 'QCD',
    'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf, t\bar{t}+cc$',
    'SingleTop': 'single top',
    'TTX': r'$t\bar{t}t\bar{t}, t\bar{t}W, t\bar{t}HW, t\bar{t}Hq$',
    'ttZ': r'$t\bar{t}Z$',
    'ttH': r'$t\bar{t}H$',
    'data_obs': 'Data'
}

# The remaining colors from your list if you need them later:
# '#e76300', '#b9ac70', '#717581', '#92dadd'

# Binning (bins, min, max)
binning_dict = {
    'nPV': (50, 0, 100), 'nPVGood': (50, 0, 100),
    'MET_pt': (40, 0, 800), 'MET_phi': (30, -3.14, 3.14),
    'lep_pt': (40, 0, 800), 'ele_pt': (40, 0, 800), 'muon_pt': (40, 0, 800),
    'lep_eta': (30, -2.5, 2.5), 'ele_eta': (30, -2.5, 2.5), 'muon_eta': (30, -2.4, 2.4),
    'n_ak4': (15, 0, 15), 'n_bjet': (10, 0, 10), 'n_ak8': (5, 0, 5),
    'jet1_pt': (40, 0, 1000), 'jet2_pt': (40, 0, 800),
    'bjet1_pt': (40, 0, 800), 'bjet2_pt': (40, 0, 600),
    'jet1_eta': (30, -2.5, 2.5), 'jet2_eta': (30, -2.5, 2.5),
    'bjet1_eta': (30, -2.5, 2.5), 'bjet2_eta': (30, -2.5, 2.5),
    'jet1_btag': (20, 0, 1), 'jet2_btag': (20, 0, 1),
    'bjet1_btag': (20, 0, 1), 'bjet2_btag': (20, 0, 1),
    'fatjet1_pt': (40, 200, 1200), 'fatjet1_eta': (30, -2.5, 2.5), 'fatjet1_mass': (40, 0, 400)
}
default_binning = (40, 0, 500)

In [122]:
def getZhbbWeight(df_, year=None):
    """
    Calculates total weight. Uses .get() and .fillna(1.0) to safely handle 
    cases where certain SFs might not exist for specific background processes.
    """
    if 'norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index) # Fallback for Data

    weight = df_['norm_weight'].copy()
    
    # Handle genWeight and topptWeight safely
    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    #top_w = df_.get('topptWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)# * top_w
    
    # Apply Scale Factors and Pileup Weight
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'puWeight']#, 'topptWeight']
    for sf in sfs:
        sf_col = df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        weight *= sf_col
        
    return weight

# Initialize a dictionary to hold lists of DataFrames for each process
temp_data_dict = {proc: [] for proc in all_processes}
variation = 'nominal'

# Include your new weight variables in the extraction list
weight_vars = ['genWeight', 'topptWeight', 'ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'puWeight']
vars_to_extract = validation_vars + ['norm_weight'] + weight_vars

coffea_files = glob.glob(os.path.join(COFFEA_DIR, "*.coffea"))
print(f"Found {len(coffea_files)} .coffea files to process.")

for file_path in coffea_files:
    filein = load(file_path)
    genweight_dict = filein.get('sum_signOf_genweights', {})
    
    for raw_proc in filein['columns'].keys():
        # 1. Map the raw process name from the coffea file to our plotting categories
        mapped_proc = None
        if 'tt+B' in raw_proc: 
            mapped_proc = 'tt_B'
        elif 'tt+LF' in raw_proc or 'tt+C' in raw_proc: 
            mapped_proc = 'TTBar'
        elif 'WJets' in raw_proc or 'DYJets' in raw_proc: 
            mapped_proc = 'VJets'
        elif 'QCD' in raw_proc: 
            mapped_proc = 'QCD'
        elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): # <-- FIXED
            mapped_proc = 'ttH'
        elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower(): # <-- CLEANED UP
            mapped_proc = 'ttZ'
        elif 'DATA' in raw_proc or 'data' in raw_proc.lower(): 
            mapped_proc = 'data_obs'
        elif 'SingleTop' in raw_proc:
            mapped_proc = 'SingleTop'
        elif 'TTX' in raw_proc:
            mapped_proc = 'TTX'
        
        if mapped_proc not in all_processes: continue
            
        for dataset in filein['columns'][raw_proc].keys():
            genweight = genweight_dict.get(dataset, 1.0) 
            if isinstance(genweight, dict):
                genweight = genweight.get(dataset, 1.0)
                
            try:
                base_dict = filein['columns'][raw_proc][dataset]['btag_mask'][variation]
            except KeyError:
                continue 
                
            tmp_data = {}
            for var in vars_to_extract:
                dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                    
                if dict_key in base_dict:
                    arr = np.array(base_dict[dict_key].value)
                    # We still do the initial norm_weight calculation here
                    if var == 'norm_weight' and mapped_proc != 'data_obs':
                        tmp_data[var] = arr / genweight
                    else:
                        tmp_data[var] = arr
            
            if tmp_data:
                temp_data_dict[mapped_proc].append(pd.DataFrame(tmp_data))

# Concatenate and apply the final weight function
data_dict = {}
for proc in all_processes:
    if temp_data_dict[proc]:
        df = pd.concat(temp_data_dict[proc], ignore_index=True)
        
        if 'n_b_outZH' in df.columns:
            df = df[df['n_b_outZH'] >= 2] 
        df = df[df['n_ak4jets'] >=5]
        data_dict[proc] = df
        
        if proc != 'data_obs':
            data_dict[proc]['tot_weight'] = getZhbbWeight(data_dict[proc], year=2024)
        else:
            data_dict[proc]['tot_weight'] = 1.0
            
        print(f"Successfully compiled {proc} ({len(data_dict[proc])} events)")
    else:
        data_dict[proc] = pd.DataFrame()

Found 5 .coffea files to process.
Successfully compiled VJets (3915 events)
Successfully compiled tt_B (653025 events)
Successfully compiled TTBar (2116151 events)
Successfully compiled SingleTop (106737 events)
Successfully compiled TTX (1278360 events)
Successfully compiled ttZ (50666 events)
Successfully compiled ttH (584860 events)
Successfully compiled data_obs (80880 events)


In [ ]:
def load_and_cut_data(variation='nominal', coffea_dir=COFFEA_DIR, year=2024):
    """
    Extracts data for a specific systematic variation, applies baseline cuts, 
    and calculates total weights. Safely falls back to 'nominal' if a process 
    (like Data) does not contain the requested systematic.
    """
    temp_data_dict = {proc: [] for proc in all_processes}
    
    coffea_files = glob.glob(os.path.join(coffea_dir, "*.coffea"))
    print(f"Extracting '{variation}' from {len(coffea_files)} files...")

    for file_path in coffea_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            # 1. Map the raw process name
            mapped_proc = None
            if 'tt+B' in raw_proc: 
                mapped_proc = 'tt_B'
            elif 'tt+LF' in raw_proc or 'tt+C' in raw_proc: 
                mapped_proc = 'TTBar'
            elif 'WJets' in raw_proc or 'DYJets' in raw_proc: 
                mapped_proc = 'VJets'
            elif 'QCD' in raw_proc: 
                mapped_proc = 'QCD'
            elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): 
                mapped_proc = 'ttH'
            elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower(): 
                mapped_proc = 'ttZ'
            elif 'DATA' in raw_proc or 'data' in raw_proc.lower(): 
                mapped_proc = 'data_obs'
            elif 'SingleTop' in raw_proc:
                mapped_proc = 'SingleTop'
            elif 'TTX' in raw_proc:
                mapped_proc = 'TTX'
            
            if mapped_proc not in all_processes: 
                continue
                
            for dataset in filein['columns'][raw_proc].keys():
                # 2. Extract Genweight
                genweight = genweight_dict.get(dataset, 1.0) 
                if isinstance(genweight, dict):
                    genweight = genweight.get(dataset, 1.0)
                    
                # 3. Handle Systematic Variation safely
                try:
                    # Try to pull the requested systematic
                    base_dict = filein['columns'][raw_proc][dataset]['btag_mask'][variation]
                except KeyError:
                    # If it doesn't exist (e.g., Data doesn't have JES Up), fallback to nominal
                    if variation != 'nominal':
                        try:
                            base_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                        except KeyError:
                            continue # Even nominal is missing, skip
                    else:
                        continue 
                    
                # 4. Extract Variables
                tmp_data = {}
                for var in vars_to_extract:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                        
                    if dict_key in base_dict:
                        arr = np.array(base_dict[dict_key].value)
                        # We still do the initial norm_weight calculation here
                        if var == 'norm_weight' and mapped_proc != 'data_obs':
                            tmp_data[var] = arr / genweight
                        else:
                            tmp_data[var] = arr
                
                if tmp_data:
                    temp_data_dict[mapped_proc].append(pd.DataFrame(tmp_data))

    # 5. Concatenate, Apply Cuts, and Apply Final Weights
    data_dict = {}
    for proc in all_processes:
        if temp_data_dict[proc]:
            df = pd.concat(temp_data_dict[proc], ignore_index=True)
            
            # --- Apply your phase space cuts here ---
            if 'n_b_outZH' in df.columns:
                df = df[df['n_b_outZH'] >= 2] 
            if 'n_ak4jets' in df.columns:
                df = df[df['n_ak4jets'] >= 5]
            
            # --- Apply total weight ---
            if proc != 'data_obs':
                df['tot_weight'] = getZhbbWeight(df, year=year)
            else:
                df['tot_weight'] = 1.0
                
            data_dict[proc] = df
        else:
            data_dict[proc] = pd.DataFrame()

    return data_dict

In [106]:
print(len(data_dict['data_obs']))
a= sum(data_dict['TTBar']['tot_weight'])
b= sum(data_dict['tt_B']['tot_weight'])
c= sum(data_dict['ttH']['tot_weight'])
d= sum(data_dict['ttZ']['tot_weight'])
e= sum(data_dict['VJets']['tot_weight'])
print(a+b+c+d+e)

80880
68649.26313632922


In [123]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages

def plot_variables_to_pdf(var_names, data_dict, output_filename="Data_MC_Plots.pdf"):
    """
    Plots a list of variables in a 3x3 grid per page and saves to a PDF.
    """
    plots_per_page = 9
    rows, cols = 3, 3
    
    # Combine backgrounds and signals for the stack
    mc_processes = bkg_processes + sig_processes
    mc_colors = {**bkg_colors, **sig_colors}

    with PdfPages(output_filename) as pdf:
        # Loop over variables in chunks of 9
        for chunk_start in range(0, len(var_names), plots_per_page):
            chunk_vars = var_names[chunk_start : chunk_start + plots_per_page]
            
            # Create a large figure for the 3x3 grid
            fig = plt.figure(figsize=(24, 24))
            outer_grid = fig.add_gridspec(rows, cols, wspace=0.3, hspace=0.3)
            
            for idx, var_name in enumerate(chunk_vars):
                row = idx // cols
                col = idx % cols
                
                # Create the inner 2x1 grid for Main and Ratio panels
                inner_grid = outer_grid[row, col].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.00)
                ax = fig.add_subplot(inner_grid[0])
                rax = fig.add_subplot(inner_grid[1], sharex=ax)
                ax.tick_params(labelbottom=False)
                
                # --- Binning Setup ---
                if var_name not in binning_dict:
                    n_bins, x_min, x_max = default_binning
                else:
                    n_bins, x_min, x_max = binning_dict[var_name]
                    
                bins = np.linspace(x_min, x_max, n_bins + 1)
                bin_centers = 0.5 * (bins[1:] + bins[:-1])
                
                # --- Process All MC (Background + Signal) ---
                mc_hists, mc_labels, mc_colors_list = [], [], []
                total_mc_counts = np.zeros(n_bins)
                total_mc_err2 = np.zeros(n_bins) 
                
                for proc in mc_processes:
                    df = data_dict.get(proc, pd.DataFrame())
                    if df.empty or var_name not in df.columns: 
                        continue
                        
                    mask = df[var_name].notna()
                    vals = df[var_name][mask]
                    if proc == 'tt_B':
                        weights = df['tot_weight'][mask] * 1.3
                    #if proc == 'VJets':
                    #    weights = df['tot_weight'][mask] * 1.21
                    else:
                        weights = df['tot_weight'][mask]

                    vals = np.clip(vals, None, bins[-1])
                    
                    counts, _ = np.histogram(vals, bins=bins, weights=weights)
                    err2, _ = np.histogram(vals, bins=bins, weights=weights**2)
                    
                    yield_total = np.sum(counts)
                    stat_unc = np.sqrt(np.sum(err2))
                    base_label = process_labels.get(proc, proc)
                    
                    # Formats as "Label (123.4 ± 5.6)"
                    label_with_yield = f"{base_label} ({yield_total:.1f} ± {stat_unc:.1f})"
                    
                    mc_hists.append(counts)
                    mc_labels.append(label_with_yield)
                    mc_colors_list.append(mc_colors[proc])
                    
                    total_mc_counts += counts
                    total_mc_err2 += err2

                # Plot all MC together in one stacked fill
                if mc_hists:
                    hep.histplot(mc_hists, bins=bins, ax=ax, stack=True, histtype='fill', 
                                 label=mc_labels, color=mc_colors_list, sort='yield')

                # Calculate and plot total MC stat uncertainty
                total_mc_err = np.sqrt(total_mc_err2)
                ax.stairs(
                    values=total_mc_counts + total_mc_err,
                    baseline=total_mc_counts - total_mc_err,
                    edges=bins, fill=True, hatch='////', edgecolor='red', 
                    facecolor='none', label='MC Stat. Unc.'
                )

                # --- Process Data ---
                data_df = data_dict.get(data_process, pd.DataFrame())
                data_counts = np.zeros(n_bins)
                data_err = np.zeros(n_bins) # Properly initialized to avoid UnboundLocalError
                
                if not data_df.empty and var_name in data_df.columns:
                    mask = data_df[var_name].notna()
                    vals = data_df[var_name][mask]
                    weights = data_df['tot_weight'][mask]

                    vals = np.clip(vals, None, bins[-1])
                    
                    data_counts, _ = np.histogram(vals, bins=bins, weights=weights)
                    data_err = np.sqrt(data_counts)

                    data_yield = np.sum(data_counts)
                    data_stat_unc = np.sqrt(data_yield)
                    base_data_label = process_labels.get(data_process, 'Data')

                    data_label_with_yield = f"{base_data_label} ({data_yield:.0f} ± {data_stat_unc:.1f})"
                    
                    hep.histplot(data_counts, bins=bins, ax=ax, stack=False, histtype='errorbar', 
                                 color='black', label=data_label_with_yield, yerr=data_err)

                # --- Ratio Panel (Data / Total MC) ---
                with np.errstate(divide='ignore', invalid='ignore'):
                    ratio = data_counts / total_mc_counts
                    ratio_err = data_err / total_mc_counts 
                    mc_rel_err = total_mc_err / total_mc_counts

                # Clean up NaNs and infs
                ratio[np.isnan(ratio) | np.isinf(ratio)] = 0
                ratio_err[np.isnan(ratio_err) | np.isinf(ratio_err)] = 0
                mc_rel_err[np.isnan(mc_rel_err) | np.isinf(mc_rel_err)] = 0

                # Ratio MC uncertainty band
                rax.stairs(
                    values=1 + mc_rel_err,
                    baseline=1 - mc_rel_err,
                    edges=bins, fill=True, hatch='////', edgecolor='red', 
                    facecolor='none'
                )

                # Ratio data points
                rax.errorbar(bin_centers, ratio, yerr=ratio_err, fmt='ko', markersize=3)
                rax.axhline(1, color='black', linestyle='--')
                
                # --- Styling for individual subplots ---
                ax.set_ylabel("Events")
                ax.legend(loc='upper right', ncol=2, fontsize=10) 
                
                if 'pt' in var_name or 'mass' in var_name: 
                    ax.set_yscale('log')
                    # Prevent log(0) issues by finding the min non-zero value or setting a hard limit
                    max_val = max(np.max(total_mc_counts), np.max(data_counts))
                    ax.set_ylim(0.1, max_val * 100 if max_val > 0 else 100)
                else:
                    max_val = max(np.max(total_mc_counts), np.max(data_counts))
                    ax.set_ylim(0.1, max_val * 100 if max_val > 0 else 100)
                    #ax.set_ylim(0, max_val * 1.5 if max_val > 0 else 10)

                ax.set_yscale('log')
                rax.set_xlabel(var_name)
                rax.set_ylabel("Data / MC")
                rax.set_ylim(0.5, 1.5)
                
                # Add CMS label to each subplot
                hep.cms.label("Preliminary", data=not data_df.empty, lumi=LUMI, ax=ax, com=13.6, fontsize=12)

            # Save the current figure (up to 9 plots) as one page in the PDF
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig) 
            
    print(f"Finished generating plots. Saved to {output_filename}")

In [124]:
# To test without getting overwhelmed, slice the list (e.g., validation_vars[:3])

plot_variables_to_pdf(validation_vars[:-2], data_dict)

Finished generating plots. Saved to Data_MC_Plots.pdf
